In [ ]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import json

load_dotenv(override=True)
openai = OpenAI()

messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is the capital of Karnataka?"}
]

system_prompt = "You are a helpful assistant"

def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    return f"Email received: {email}, and recorded successfully!"

record_email_tool_json = {
    "type": "function",
    "function": {
        "name": "record_email_tool",
        "description": "Use this tool to record that a user provided their email address",
        "parameters": {
            "type": "object",
            "properties": {
                "email": {"type": "string", "description": "The email address of this user"}
            },
            "required": ["email"],
            "additionalProperties": False
        }
    }
}

def add_number_tool(num1, num2):
    return num1 + num2

add_number_tool_json = {
  "type": "function",
  "function": {
    "name": "add_number_tool",
    "description": "",
    "parameters": {
      "type": "object",
      "properties": {
        "num1": {
          "type": "string"
        },
        "num2": {
          "type": "string"
        }
      },
      "required": [
        "num1",
        "num2"
      ],
      "additionalProperties": False
    },
    "strict": True
  }
}


tools = [ record_email_tool_json, add_number_tool_json]

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
         
    while response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            messages.append(message)
            for tool_call in message.tool_calls:
                email = json.loads(tool_call.function.arguments).get("email")
                record_email_tool(email)
                messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
            
    return response.choices[0].message.content

In [ ]:
# messsage = "please record email : john@duk.com, and also what is capital of Karnataka?"
# print(chat(messsage, []))

In [ ]:
demo = gr.ChatInterface(chat)
demo.launch(inbrowser=True)

In [ ]:
demo.close()